# ProjectCerebro Week 2 ML Training on Google Colab

Use a T4 GPU runtime. Before running this notebook, add a Google Drive shortcut from **Shared with me** to one of these locations:

- `My Drive/ProjectCerebro-Shared`
- `My Drive/projectcerebro/ProjectCerebro-Shared`

This fixed notebook auto-detects the Drive shortcut, installs Spark/Delta explicitly, keeps Colab CUDA PyTorch, and trains from the shared Drive `delta_lake/` folder.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Clone repo and switch branch

This requires `chiranjeev/ml-core-week2` to be pushed to GitHub first.

In [ ]:
%cd /content
!rm -rf projectcerebro
!git clone https://github.com/JANARDHANAREDDYMS/projectcerebro.git
%cd /content/projectcerebro
!git fetch origin
!git switch chiranjeev/ml-core-week2

## 3. Set and verify Drive data paths

In [ ]:
import os
from pathlib import Path

candidate_roots = [
    "/content/drive/MyDrive/ProjectCerebro-Shared/delta_lake",
    "/content/drive/MyDrive/projectcerebro/ProjectCerebro-Shared/delta_lake",
]

DATA_ROOT = None
for candidate in candidate_roots:
    root = Path(candidate)
    bp8 = root / "epochs_mi_v1_ch5_sr128_bp8_30"
    bp4 = root / "epochs_mi_v1_ch5_sr128_bp4_38"
    print(candidate, "exists=", root.exists(), "bp8_delta_log=", (bp8 / "_delta_log").exists())
    if (bp8 / "_delta_log").exists() and (bp4 / "_delta_log").exists():
        DATA_ROOT = str(root)
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find ProjectCerebro Delta Lake folders. Add the ProjectCerebro-Shared "
        "shortcut to My Drive, then Runtime -> Disconnect and delete runtime, reconnect, and rerun."
    )

BP8_30 = f"{DATA_ROOT}/epochs_mi_v1_ch5_sr128_bp8_30"
BP4_38 = f"{DATA_ROOT}/epochs_mi_v1_ch5_sr128_bp4_38"

os.environ["DATA_ROOT"] = DATA_ROOT
os.environ["BP8_30"] = BP8_30
os.environ["BP4_38"] = BP4_38

print("\nUsing DATA_ROOT:", DATA_ROOT)
for path in [BP8_30, BP4_38]:
    print(path, "exists=", Path(path).exists(), "delta_log=", Path(path, "_delta_log").exists())


In [ ]:
!ls "$BP8_30"
!ls "$BP8_30/_delta_log" | head

## 4. Install dependencies without replacing Colab CUDA PyTorch

This cell intentionally removes `torch`, `pyspark`, and `delta_spark` from the repo requirements first. Colab already has CUDA PyTorch, and Spark/Delta are installed explicitly with package names that expose the `delta` Python module.


In [ ]:
!grep -Ev "^(torch|pyspark|delta[_-]spark)==|^appnope==" requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
!pip install -q "pyspark==4.1.1" "delta-spark==4.2.0" "pyarrow==15.0.0"


In [ ]:
import importlib.util

for module_name in ["torch", "pyspark", "delta", "pyarrow"]:
    spec = importlib.util.find_spec(module_name)
    print(module_name, "OK" if spec else "MISSING")
    assert spec is not None, f"Missing required module: {module_name}"

from delta import configure_spark_with_delta_pip
print("Delta Lake Python package is ready")


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## 5. Run unit tests

In [ ]:
!pytest tests -v

## 6. Smoke training

In [ ]:
!python -m ml_core.experiments.train_smoke \
  --filter bp8_30 \
  --delta-path "$BP8_30"

## 7. ShallowConvNet baseline

In [ ]:
!python -m ml_core.experiments.train_shallow_baseline \
  --filter bp8_30 \
  --delta-path "$BP8_30"

## 8. EEGNet PhysioNet pretrain

In [ ]:
!python -m ml_core.experiments.pretrain_eegnet_physionet \
  --filter bp8_30 \
  --delta-path "$BP8_30"

## 9. EEGNet BCI fine-tune

In [ ]:
!python -m ml_core.experiments.finetune_eegnet_bci \
  --filter bp8_30 \
  --delta-path "$BP8_30" \
  --pretrained artifacts/checkpoints/eegnet_pretrain_bp8_30/best.pt

## 10. Inspect and persist artifacts

In [ ]:
!find artifacts/checkpoints -maxdepth 3 -type f | sort
!find artifacts/mlruns -maxdepth 3 -type f | head -50

In [ ]:
!mkdir -p "/content/drive/MyDrive/projectcerebro/training_artifacts"
!cp -r artifacts/checkpoints artifacts/reports artifacts/mlruns "/content/drive/MyDrive/projectcerebro/training_artifacts/"
!find "/content/drive/MyDrive/projectcerebro/training_artifacts" -maxdepth 2 -type d | sort
